# Лабораторная работа 9. Вариант 1
**Тема.** Open-shop задачи

## **Задача 1**
$$
O_2 | \space | C_{max}
$$ 
Задачи типа **open-shop** характеризуются свободным порядком выполнения работ. 

### Алгоритм
Решение здесь находится через LAPT [1, 8.1]. Суть LAPT:
> Приоритет отдается работам с наибольшей суммарной длительностью выполнения на всех станках

Для каждой работы считается $P_j$:
$$
P_j = p_{j1}+p_{j2}
$$
после чего работы упорядочиваются по $P_j$

In [1]:
jobs = {
    'J1': {'p1': 1, 'p2': 13},
    'J2': {'p1': 10, 'p2': 12},
    'J3': {'p1': 17, 'p2': 9},
    'J4': {'p1': 12, 'p2': 17},
    'J5': {'p1': 11, 'p2': 3}
}

In [2]:
done = {j_id: {1: False, 2: False} for j_id in jobs}
m_free = {1: 0, 2: 0}
j_busy_until = {j_id: 0 for j_id in jobs}

schedule = []
total_ops = len(jobs) * 2
completed_ops = 0

current_time = 0

Проведем моделирование

In [3]:
while completed_ops < total_ops:
    for m_id in [1, 2]:
        if m_free[m_id] <= current_time:
            other_m = 2 if m_id == 1 else 1
            candidates = []
            for j_id, times in jobs.items():
                if not done[j_id][m_id] and j_busy_until[j_id] <= current_time:
                    weight = times[f'p{other_m}']
                    candidates.append((weight, j_id))
            
            if candidates:
                candidates.sort(key=lambda x: x[0], reverse=True)
                _, best_j = candidates[0]
                
                p_curr = jobs[best_j][f'p{m_id}']
                start = current_time
                end = start + p_curr
                
                m_free[m_id] = end
                j_busy_until[best_j] = end
                done[best_j][m_id] = True
                completed_ops += 1
                schedule.append((best_j, m_id, start, end))
    
    current_time = min(m_free.values()) if any(not d[1] or not d[2] for d in done.values()) else current_time + 1
    if current_time > 1000: break

Выведем результат

In [4]:
plan, c_max = schedule, max(m_free.values())

print(f"{'Работа':<7} | {'Станок':<7} | {'Начало':<7} | {'Конец'}")
print("-" * 40)
for j, m, s, e in sorted(plan, key=lambda x: x[2]):
    print(f"{j:<7} | M{m:<6} | {s:<7} | {e}")
print("-" * 40)
print(f"Итоговый C_max: {c_max}")

Работа  | Станок  | Начало  | Конец
----------------------------------------
J4      | M1      | 0       | 12
J3      | M2      | 0       | 9
J5      | M2      | 9       | 12
J1      | M1      | 12      | 13
J4      | M2      | 12      | 29
J2      | M1      | 13      | 23
J3      | M1      | 23      | 40
J2      | M2      | 29      | 41
J5      | M1      | 40      | 51
J1      | M2      | 41      | 54
----------------------------------------
Итоговый C_max: 54


## **Задача 2**
$$
O_3 | \space | C_{max}
$$ 
По аналогии со всеми предыдущими классами задач, open shop с тремя и более станками становится NP-сложной.

### Алгоритм 
Здесь для решения применяется эвристика LTRP-OM (Longest Total Remaining Processing on Other Machines first rule) [1, 8.1]. Правило формулируется следующим образом:
> В каждый момент времени $t$, когда станок $M_i$ освобождается, он выбирает среди доступных работ ту, у которой максимальная суммарная длительность операций на остальных станках.

Формулой приоритета для станка $M_1$ будет:
$$
W_j = p_{j2}+p_{j3}
$$

In [5]:
jobs = {
    'J1': {1: 1, 2: 13, 3: 6},
    'J2': {1: 10, 2: 12, 3: 18},
    'J3': {1: 17, 2: 9, 3: 13},
    'J4': {1: 12, 2: 17, 3: 2},
    'J5': {1: 11, 2: 3, 3: 5}
}

In [6]:
all_machines = sorted(list(set(m for j in jobs.values() for m in j.keys())))
done = {j_id: {m_id: (m_id not in jobs[j_id]) for m_id in all_machines} for j_id in jobs}

m_free = {m_id: 0 for m_id in all_machines}
j_busy_until = {j_id: 0 for j_id in jobs}

schedule = []
total_ops = sum(len(j_ops) for j_ops in jobs.values())
completed_ops = 0
current_time = 0

Проведем моделирование

In [7]:
while completed_ops < total_ops:
    progress = False
    available_machines = [m for m in all_machines if m_free[m] <= current_time]
    
    for m_id in available_machines:
        candidates = []
        for j_id, p_times in jobs.items():
            if m_id in p_times and not done[j_id][m_id] and j_busy_until[j_id] <= current_time:
                weight = sum(p_times[m] for m in p_times if m != m_id and not done[j_id][m])
                candidates.append((weight, j_id))
        
        if candidates:
            candidates.sort(key=lambda x: x[0], reverse=True)
            _, best_j = candidates[0]
            
            duration = jobs[best_j][m_id]
            start = current_time
            end = start + duration
            
            m_free[m_id] = end
            j_busy_until[best_j] = end
            done[best_j][m_id] = True
            completed_ops += 1
            schedule.append({'j': best_j, 'm': m_id, 's': start, 'e': end})
            progress = True

    events = [t for t in list(m_free.values()) + list(j_busy_until.values()) if t > current_time]
    if events:
        current_time = min(events)
    else:
        if completed_ops < total_ops: current_time += 1
        else: break

Тогда результат будет:

In [8]:
plan, final_cmax = schedule, max(m_free.values())

print(f"{'Работа':<7} | {'Станок':<7} | {'Начало':<7} | {'Конец'}")
print("-" * 40)
for op in sorted(plan, key=lambda x: (x['s'], x['m'])):
    print(f"{op['j']:<7} | M{op['m']:<6} | {op['s']:<7} | {op['e']}")
print("-" * 40)
print(f"C_max (LTRP-OM): {final_cmax}")

Работа  | Станок  | Начало  | Конец
----------------------------------------
J2      | M1      | 0       | 10
J3      | M2      | 0       | 9
J4      | M3      | 0       | 2
J1      | M3      | 2       | 8
J5      | M3      | 8       | 13
J4      | M2      | 9       | 26
J1      | M1      | 10      | 11
J3      | M1      | 11      | 28
J2      | M3      | 13      | 31
J5      | M2      | 26      | 29
J4      | M1      | 28      | 40
J1      | M2      | 29      | 42
J3      | M3      | 31      | 44
J5      | M1      | 40      | 51
J2      | M2      | 42      | 54
----------------------------------------
C_max (LTRP-OM): 54


## **Задача 3**
$$
O_3 | prmp | C_{max}
$$ 
### Алгоритм
Когда в задаче Open job появляются прерывания одним из варинатов ее решения становится алгоритм Биркгофа — фон Неймана. Он основан на теории матриц и рассматривает задачу как операции на квадратичных матриц.

In [9]:
jobs = {
    'J1': {1: 1, 2: 13, 3: 6},
    'J2': {1: 10, 2: 12, 3: 18},
    'J3': {1: 17, 2: 9, 3: 13},
    'J4': {1: 12, 2: 17, 3: 2},
    'J5': {1: 11, 2: 3, 3: 5}
}

**Поиск совершенного паросочетания.** На каждой итерации нам нужно выбрать такой набор операций, чтобы:
- Каждый станок был занят ровно одной работой.
- Каждая работа находилась ровно на одном станке.

In [10]:
import numpy as np
def find_perfect_matching(adj):
    n = adj.shape[0]
    match = [-1] * n
    def dfs(u, visited):
        for v in range(n):
            if adj[u, v] and not visited[v]:
                visited[v] = True
                if match[v] < 0 or dfs(match[v], visited):
                    match[v] = u
                    return True
        return False

    for i in range(n):
        visited = [False] * n
        if not dfs(i, visited): return None
    
    res = [0] * n
    for v, u in enumerate(match): res[u] = v
    return res

Проведем полный ход алгоритма

In [11]:
job_keys = sorted(jobs.keys())
machine_keys = [1, 2, 3]
raw_matrix = np.zeros((len(machine_keys), len(job_keys)))
for j_idx, j_key in enumerate(job_keys):
    for m_idx, m_key in enumerate(machine_keys):
        raw_matrix[m_idx, j_idx] = jobs[j_key][m_key]
        
m_sums = np.sum(raw_matrix, axis=1)
j_sums = np.sum(raw_matrix, axis=0)
T = max(np.max(m_sums), np.max(j_sums))
size = max(raw_matrix.shape)
full_matrix = np.zeros((size, size))
full_matrix[:raw_matrix.shape[0], :raw_matrix.shape[1]] = raw_matrix
for i in range(size):
    for j in range(size):
        needed_in_row = T - np.sum(full_matrix[i, :])
        needed_in_col = T - np.sum(full_matrix[:, j])
        if needed_in_row > 0 and needed_in_col > 0:
            fill = min(needed_in_row, needed_in_col)
            full_matrix[i, j] += fill

remaining = full_matrix.copy()
schedule = []

while np.any(remaining > 1e-7):
    adj = (remaining > 1e-7).astype(int)
    perm = find_perfect_matching(adj)
    if perm is None: break
    delta = min(remaining[i, j] for i, j in enumerate(perm))
    current_ops = []
    for m_idx, j_idx in enumerate(perm):
        if m_idx < 3:
            real_p = raw_matrix[m_idx, j_idx] if j_idx < len(job_keys) else 0
            current_ops.append((m_idx + 1, job_keys[j_idx] if j_idx < len(job_keys) else "Idle"))
    
    schedule.append((delta, current_ops))
    for i, j in enumerate(perm):
        remaining[i, j] -= delta

Полученные результаты:

In [12]:
target_c_max, final_schedule = T, schedule
print(f"Оптимальный C_max: {target_c_max}")
print("-" * 60)
time = 0
for delta, ops in final_schedule:
    print(f"[{time:4.1f} - {time+delta:4.1f}] | Длительность: {delta:4.1f}")
    for m, j in sorted(ops):
        status = f"делает {j}" if j != "Idle" else "простаивает"
        print(f"  Станок M{m}: {status}")
    time += delta
    print("-" * 60)

Оптимальный C_max: 54.0
------------------------------------------------------------
[ 0.0 -  3.0] | Длительность:  3.0
  Станок M1: делает J1
  Станок M2: делает J5
  Станок M3: делает J3
------------------------------------------------------------
[ 3.0 -  4.0] | Длительность:  1.0
  Станок M1: делает J1
  Станок M2: делает J3
  Станок M3: делает J5
------------------------------------------------------------
[ 4.0 -  8.0] | Длительность:  4.0
  Станок M1: делает J2
  Станок M2: делает J3
  Станок M3: делает J5
------------------------------------------------------------
[ 8.0 - 12.0] | Длительность:  4.0
  Станок M1: делает J5
  Станок M2: делает J3
  Станок M3: делает J2
------------------------------------------------------------
[12.0 - 19.0] | Длительность:  7.0
  Станок M1: делает J5
  Станок M2: делает J1
  Станок M3: делает J3
------------------------------------------------------------
[19.0 - 32.0] | Длительность: 13.0
  Станок M1: делает J3
  Станок M2: делает J4
  Станок 

## References
1. Scheduling. Theory, Algorithms, and Systems., Michael L. Pinedo